# 6-Model Ensemble Validation
## Reference Period: 1995–2014 | SSP2-4.5

**Purpose:**  
Evaluate the performance of the 6-model equal-weight ensemble against observed precipitation climatology using the same five statistical metrics applied in Notebooks 03 and 07. Results complete the three-way comparison across all modelling approaches examined in the manuscript

| Approach | Notebook | Description |
|----------|----------|-------------|
| Best single model | 03 | Top-ranked model per basin from composite ranking |
| 3-model ensemble | 07 | Arithmetic mean of top-3 ranked models per basin |
| **6-model ensemble** | **10 (this)** | **Arithmetic mean of all 6 models — every cell** |

The systematic deterioration (or improvement) in RMSE, MAE, and PBIAS across ensemble size directly addresses the ensemble methodology findings of the manuscript.

**Outputs saved to:**  
`C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\ensemble.6model\`


## 1. Import Libraries

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

print("Libraries loaded.")
print(f"  numpy  : {np.__version__}")
print(f"  pandas : {pd.__version__}")


Libraries loaded.
  numpy  : 2.1.3
  pandas : 2.2.3


## 2. Configuration

In [2]:
# ── Input paths ───────────────────────────────────────────────────────────────
OBS_LTMM_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\stations data\long_term_monthly_mean\obs_ltmm_all_stations.csv"
)

ENS6_LTMM_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\6ensemble data\long_term_monthly_mean\ltmm_6ens_all_stations.csv"
)

STATION_MAPPING_FILE = Path(
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS"
    r"\Station_Basin_Mapping.xlsx"
)

# Previous validation results for cross-approach comparison
SINGLE_TABLE3_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\validation\single.model\table3_best_model_per_basin.csv"
)
ENS3_BASIN_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\validation\ensemble.3model\basin_level_metrics_ensemble.csv"
)
ENS3_STN_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\validation\ensemble.3model\station_level_metrics_ensemble.csv"
)

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\validation\ensemble.6model"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Obs LTMM       : {OBS_LTMM_CSV}")
print(f"  6-ens LTMM     : {ENS6_LTMM_CSV}")
print(f"  Output         : {OUTPUT_ROOT}")


Configuration loaded.
  Obs LTMM       : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\stations data\long_term_monthly_mean\obs_ltmm_all_stations.csv
  6-ens LTMM     : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6ensemble data\long_term_monthly_mean\ltmm_6ens_all_stations.csv
  Output         : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\ensemble.6model


## 3. Statistical Performance Metrics

Identical metric functions to Notebooks 03 and 07 — self-contained for independent execution.


In [3]:
def compute_metrics(obs: np.ndarray, mod: np.ndarray) -> dict:
    """
    Compute five evaluation metrics (r, NSE, PBIAS, RMSE, MAE).
    NaN pairs are excluded before computation. Requires n >= 3.
    """
    obs  = np.asarray(obs, dtype=float)
    mod  = np.asarray(mod, dtype=float)
    mask = np.isfinite(obs) & np.isfinite(mod)
    obs, mod = obs[mask], mod[mask]
    n = len(obs)

    if n < 3:
        return {"r": np.nan, "NSE": np.nan, "PBIAS": np.nan,
                "RMSE": np.nan, "MAE": np.nan, "N": n}

    obs_mean = np.mean(obs)
    num      = np.sum((obs - obs_mean) * (mod - np.mean(mod)))
    denom    = np.sqrt(np.sum((obs - obs_mean)**2) * np.sum((mod - np.mean(mod))**2))
    r        = num / denom if denom > 0 else np.nan

    ss_res   = np.sum((obs - mod)**2)
    ss_tot   = np.sum((obs - obs_mean)**2)
    nse      = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan

    pbias    = (np.sum(mod - obs) / np.sum(obs)) * 100 if np.sum(obs) != 0 else np.nan
    rmse     = np.sqrt(np.mean((mod - obs)**2))
    mae      = np.mean(np.abs(mod - obs))

    return {"r": round(r, 6), "NSE": round(nse, 6),
            "PBIAS": round(pbias, 4), "RMSE": round(rmse, 4),
            "MAE": round(mae, 4), "N": n}


print("Metric functions defined: r | NSE | PBIAS | RMSE | MAE")


Metric functions defined: r | NSE | PBIAS | RMSE | MAE


## 4. Load Observed and 6-Model Ensemble LTMM

In [4]:
obs_ltmm = pd.read_csv(OBS_LTMM_CSV)
obs_ltmm["Station_ID"] = obs_ltmm["Station_ID"].astype(str).str.strip()

ens6_ltmm = pd.read_csv(ENS6_LTMM_CSV)
ens6_ltmm["Station_ID"] = ens6_ltmm["Station_ID"].astype(str).str.strip()

stations = pd.read_excel(STATION_MAPPING_FILE)
stations["Station_ID"] = stations["Station_ID"].astype(str).str.strip()
stations["Basin"]      = stations["Basin"].astype(str).str.strip()
basin_order = list(dict.fromkeys(stations["Basin"].tolist()))

print(f"Observed LTMM   : {len(obs_ltmm):,} rows  "
      f"({obs_ltmm['Station_ID'].nunique()} stations)")
print(f"6-Ens LTMM      : {len(ens6_ltmm):,} rows  "
      f"({ens6_ltmm['Station_ID'].nunique()} stations)")
print(f"Basins          : {len(basin_order)}")


Observed LTMM   : 588 rows  (49 stations)
6-Ens LTMM      : 588 rows  (49 stations)
Basins          : 12


## 5. Station-Level Performance Metrics

The 12-month observed LTMM is compared against the 12-month 6-model ensemble LTMM at each station. One metric set per station (49 total).


In [5]:
ENSEMBLE_LABEL = "6-Model Ensemble"
station_metric_rows = []

for sid in obs_ltmm["Station_ID"].unique():
    obs_vec = (obs_ltmm[obs_ltmm["Station_ID"] == sid]
               .sort_values("Month").set_index("Month")["LTMM_mm_month"])
    ens_vec = (ens6_ltmm[ens6_ltmm["Station_ID"] == sid]
               .sort_values("Month").set_index("Month")["LTMM_mm_month"])

    aligned = obs_vec.to_frame("obs").join(ens_vec.to_frame("mod"), how="inner")
    if aligned.empty or len(aligned) < 3:
        continue

    metrics = compute_metrics(aligned["obs"].values, aligned["mod"].values)

    basin = obs_ltmm.loc[obs_ltmm["Station_ID"] == sid, "Basin"].iloc[0]
    sname = obs_ltmm.loc[obs_ltmm["Station_ID"] == sid, "Station_Name"].iloc[0]

    station_metric_rows.append({
        "Model"       : ENSEMBLE_LABEL,
        "Station_ID"  : sid,
        "Station_Name": sname,
        "Basin"       : basin,
        **metrics
    })

stn_metrics_df = pd.DataFrame(station_metric_rows)
stn_metrics_df.to_csv(OUTPUT_ROOT / "station_level_metrics_6ens.csv", index=False)

print(f"Station metrics computed : {len(stn_metrics_df)} stations")
print(f"Saved: station_level_metrics_6ens.csv")
print()
print("Overall 6-ens statistics (mean across all stations):")
print(stn_metrics_df[["r","NSE","PBIAS","RMSE","MAE"]].mean().round(4).to_string())


Station metrics computed : 49 stations
Saved: station_level_metrics_6ens.csv

Overall 6-ens statistics (mean across all stations):
r         0.9581
NSE       0.3591
PBIAS    21.0346
RMSE     10.3931
MAE       6.5328


## 6. Basin-Level Performance Metrics

In [6]:
basin_metric_rows = []

for basin in basin_order:
    grp = stn_metrics_df[stn_metrics_df["Basin"] == basin]

    if grp.empty:
        basin_metric_rows.append({
            "Basin": basin, "Model": ENSEMBLE_LABEL,
            "r": np.nan, "NSE": np.nan, "PBIAS": np.nan,
            "RMSE": np.nan, "MAE": np.nan, "N_Stations": 0
        })
        continue

    basin_metric_rows.append({
        "Basin"      : basin,
        "Model"      : ENSEMBLE_LABEL,
        "r"          : round(grp["r"].mean(),    6),
        "NSE"        : round(grp["NSE"].mean(),  6),
        "PBIAS"      : round(grp["PBIAS"].mean(),4),
        "RMSE"       : round(grp["RMSE"].mean(), 4),
        "MAE"        : round(grp["MAE"].mean(),  4),
        "N_Stations" : len(grp),
    })

basin_metrics_df = pd.DataFrame(basin_metric_rows)
basin_metrics_df.to_csv(OUTPUT_ROOT / "basin_level_metrics_6ens.csv", index=False)

print("Basin-level 6-model ensemble metrics:")
print(f"Saved: basin_level_metrics_6ens.csv")
print()
print(basin_metrics_df[["Basin","r","NSE","PBIAS","RMSE","MAE","N_Stations"]]
      .to_string(index=False))


Basin-level 6-model ensemble metrics:
Saved: basin_level_metrics_6ens.csv

                Basin        r       NSE    PBIAS    RMSE     MAE  N_Stations
              N.R.S.W 0.976502  0.887126  -6.3864 13.9834  8.1492           4
     YARMOUK (JORDAN) 0.973098  0.577025  -6.1682 11.7478  7.5572           5
JORDAN VALLY (JORDAN) 0.987569  0.074049  -0.8757 23.6076 15.9271           3
 AMMAN ZARQA (JORDAN) 0.979176  0.395755  37.1573 11.2115  7.3131           6
              S.R.S.W 0.990364  0.727701  -1.4272 15.2551  9.1255           4
            D.S.R.S.W 0.971383  0.903170  -8.6087  8.0195  4.7734           7
                MUJIB 0.960336  0.484168  31.0582  8.1920  5.2275           6
       W. ARABA NORTH 0.932893  0.787063 -16.7398  9.5501  5.1586           3
                 HASA 0.948444  0.878900  -5.2383  5.9163  3.6399           1
       AZRAQ (JORDAN) 0.900456 -1.567547 122.9500  7.1322  5.0245           4
                JAFER 0.929392 -0.090026  49.1772  4.3327  2.7927  

## 7. Summary Table 

Basin-level metrics with Mean and Std Dev summary rows, saved as both CSV and formatted Excel.


In [7]:
table_6ens = basin_metrics_df[["Basin","r","NSE","PBIAS","RMSE","MAE"]].copy()

numeric_cols = ["r","NSE","PBIAS","RMSE","MAE"]
mean_row = {"Basin": "Mean"}
std_row  = {"Basin": "Std Dev"}
for col in numeric_cols:
    vals = pd.to_numeric(table_6ens[col], errors="coerce").dropna()
    mean_row[col] = round(vals.mean(), 3)
    std_row[col]  = round(vals.std(),  3)

table_6ens_full = pd.concat(
    [table_6ens, pd.DataFrame([mean_row, std_row])],
    ignore_index=True
)
table_6ens_full.to_csv(OUTPUT_ROOT / "table_6ens_basin_metrics.csv", index=False)

try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    ws = wb.active
    ws.title = "6-Ens Basin Metrics"

    thin   = Side(style="thin", color="AAAAAA")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left   = Alignment(horizontal="left",   vertical="center")
    hdr_fill   = PatternFill("solid", start_color="145A32")   # dark green for 6-ens
    mean_fill  = PatternFill("solid", start_color="E9F7EF")
    std_fill   = PatternFill("solid", start_color="F9F9F9")
    title_font = Font(name="Arial", bold=True, size=12, color="145A32")

    ws.merge_cells("A1:F1")
    ws["A1"].value     = "6-Model Ensemble Performance Metrics Across Jordan Basins (1995-2014)"
    ws["A1"].font      = title_font
    ws["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 22

    headers    = ["Basin","Correlation (r)","NSE","PBIAS (%)","RMSE (mm/month)","MAE (mm/month)"]
    col_widths = [24, 14, 10, 12, 16, 16]
    for ci, (h, w) in enumerate(zip(headers, col_widths), 1):
        c = ws.cell(row=2, column=ci, value=h)
        c.fill = hdr_fill
        c.font = Font(name="Arial", bold=True, color="FFFFFF", size=9)
        c.alignment = center
        c.border    = border
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.row_dimensions[2].height = 32

    for ri, row in table_6ens_full.iterrows():
        r        = ri + 3
        is_stat  = row["Basin"] in ("Mean","Std Dev")
        row_fill = mean_fill if row["Basin"] == "Mean" else (
                   std_fill  if row["Basin"] == "Std Dev" else None)
        vals = [row["Basin"], row["r"], row["NSE"],
                row["PBIAS"], row["RMSE"], row["MAE"]]
        for ci, v in enumerate(vals, 1):
            cell = ws.cell(row=r, column=ci)
            cell.value     = v if not (isinstance(v, float) and np.isnan(v)) else ""
            cell.font      = Font(name="Arial", bold=is_stat, size=9)
            cell.alignment = left if ci == 1 else center
            cell.border    = border
            if row_fill:
                cell.fill  = row_fill
            if ci > 1 and isinstance(v, float) and not np.isnan(v):
                cell.number_format = "0.000"

    ws.freeze_panes = "A3"
    xl_path = OUTPUT_ROOT / "table_6ens_basin_metrics.xlsx"
    wb.save(xl_path)
    print(f"Excel saved: {xl_path}")
except Exception as e:
    print(f"Excel skipped: {e}")

print()
print("6-Model Ensemble Basin Metrics:")
print("=" * 65)
print(table_6ens_full.to_string(index=False))


Excel saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\ensemble.6model\table_6ens_basin_metrics.xlsx

6-Model Ensemble Basin Metrics:
                Basin        r       NSE    PBIAS    RMSE     MAE
              N.R.S.W 0.976502  0.887126  -6.3864 13.9834  8.1492
     YARMOUK (JORDAN) 0.973098  0.577025  -6.1682 11.7478  7.5572
JORDAN VALLY (JORDAN) 0.987569  0.074049  -0.8757 23.6076 15.9271
 AMMAN ZARQA (JORDAN) 0.979176  0.395755  37.1573 11.2115  7.3131
              S.R.S.W 0.990364  0.727701  -1.4272 15.2551  9.1255
            D.S.R.S.W 0.971383  0.903170  -8.6087  8.0195  4.7734
                MUJIB 0.960336  0.484168  31.0582  8.1920  5.2275
       W. ARABA NORTH 0.932893  0.787063 -16.7398  9.5501  5.1586
                 HASA 0.948444  0.878900  -5.2383  5.9163  3.6399
       AZRAQ (JORDAN) 0.900456 -1.567547 122.9500  7.1322  5.0245
                JAFER 0.929392 -0.090026  49.1772  4.3327  2.7927
               HAMMAD 0.864256 -0.092992  64.1568  5

## 8. Three-Way Comparison: Best Single | 3-Model Ensemble | 6-Model Ensemble

This is the central comparison table of Section 3.2 of the manuscript. Basin-level metrics for all three approaches are assembled side by side. For each metric and basin, the best-performing approach is identified.


In [8]:
rows_compare = []

for basin in basin_order:
    row = {"Basin": basin}

    # Best single model
    if SINGLE_TABLE3_CSV.exists():
        t3 = pd.read_csv(SINGLE_TABLE3_CSV)
        t3_row = t3[t3["Basin"] == basin]
        if not t3_row.empty:
            r = t3_row.iloc[0]
            row.update({
                "Single_Model": r.get("Best_Model", "?"),
                "Single_r"    : r.get("r",    np.nan),
                "Single_NSE"  : r.get("NSE",  np.nan),
                "Single_RMSE" : r.get("RMSE", np.nan),
                "Single_MAE"  : r.get("MAE",  np.nan),
                "Single_PBIAS": r.get("PBIAS",np.nan),
            })

    # 3-model ensemble
    if ENS3_BASIN_CSV.exists():
        e3 = pd.read_csv(ENS3_BASIN_CSV)
        e3_row = e3[e3["Basin"] == basin]
        if not e3_row.empty:
            r = e3_row.iloc[0]
            row.update({
                "Ens3_r"    : r.get("r",    np.nan),
                "Ens3_NSE"  : r.get("NSE",  np.nan),
                "Ens3_RMSE" : r.get("RMSE", np.nan),
                "Ens3_MAE"  : r.get("MAE",  np.nan),
                "Ens3_PBIAS": r.get("PBIAS",np.nan),
            })

    # 6-model ensemble
    e6_row = basin_metrics_df[basin_metrics_df["Basin"] == basin]
    if not e6_row.empty:
        r = e6_row.iloc[0]
        row.update({
            "Ens6_r"    : r.get("r",    np.nan),
            "Ens6_NSE"  : r.get("NSE",  np.nan),
            "Ens6_RMSE" : r.get("RMSE", np.nan),
            "Ens6_MAE"  : r.get("MAE",  np.nan),
            "Ens6_PBIAS": r.get("PBIAS",np.nan),
        })

    rows_compare.append(row)

compare_df = pd.DataFrame(rows_compare)
compare_df.to_csv(OUTPUT_ROOT / "three_approach_comparison_basin.csv", index=False)
print(f"Three-way comparison saved: three_approach_comparison_basin.csv")
print()

# Print RMSE comparison (most informative metric)
rmse_cols = [c for c in ["Single_RMSE","Ens3_RMSE","Ens6_RMSE"] if c in compare_df.columns]
print(f"{'Basin':<28} " + "  ".join(f"{c:>14}" for c in rmse_cols))
print("-" * (28 + 16 * len(rmse_cols)))
for _, row in compare_df.iterrows():
    vals = [row.get(c, np.nan) for c in rmse_cols]
    val_str = "  ".join(f"{v:>14.4f}" if not np.isnan(v) else f"{'N/A':>14}" for v in vals)
    print(f"{row['Basin']:<28} {val_str}")


Three-way comparison saved: three_approach_comparison_basin.csv

Basin                           Single_RMSE       Ens3_RMSE       Ens6_RMSE
----------------------------------------------------------------------------
N.R.S.W                             12.7923         12.8168         13.9834
YARMOUK (JORDAN)                     8.9605         10.5690         11.7478
JORDAN VALLY (JORDAN)               14.7365         23.0201         23.6076
AMMAN ZARQA (JORDAN)                10.8312         10.9467         11.2115
S.R.S.W                             14.7574         15.0550         15.2551
D.S.R.S.W                            8.0511          7.6683          8.0195
MUJIB                                7.4894          7.7968          8.1920
W. ARABA NORTH                       9.3593          9.3498          9.5501
HASA                                 4.8159          5.2912          5.9163
AZRAQ (JORDAN)                       6.1101          6.0097          7.1322
JAFER                 

## 9. Median Metric Summary Across All Three Approaches

Basin-median performance metrics for all approaches — directly comparable to Figure 2 (box plots) in the manuscript. RMSE and MAE medians quantify the systematic deterioration (or improvement) with ensemble size discussed in Section 3.2.


In [9]:
summary_rows = []

# Best single model
if SINGLE_TABLE3_CSV.exists():
    t3 = pd.read_csv(SINGLE_TABLE3_CSV)
    t3d = t3[~t3["Basin"].isin(["Mean","Std Dev"])]
    summary_rows.append({
        "Approach"     : "Best Single Model",
        "Median_r"     : round(t3d["r"].median(),         3),
        "Median_NSE"   : round(t3d["NSE"].median(),       3),
        "Median_RMSE"  : round(t3d["RMSE"].median(),      3),
        "Median_MAE"   : round(t3d["MAE"].median(),       3),
        "Median_absPBIAS": round(t3d["PBIAS"].abs().median(), 3),
    })

# 3-model ensemble
if ENS3_BASIN_CSV.exists():
    e3 = pd.read_csv(ENS3_BASIN_CSV)
    e3v = pd.to_numeric(e3["r"], errors="coerce")
    e3d = e3[e3v.notna()]
    summary_rows.append({
        "Approach"     : "3-Model Ensemble (basin-specific)",
        "Median_r"     : round(e3d["r"].median(),         3),
        "Median_NSE"   : round(e3d["NSE"].median(),       3),
        "Median_RMSE"  : round(e3d["RMSE"].median(),      3),
        "Median_MAE"   : round(e3d["MAE"].median(),       3),
        "Median_absPBIAS": round(e3d["PBIAS"].abs().median(), 3),
    })

# 6-model ensemble
e6d = basin_metrics_df[pd.to_numeric(basin_metrics_df["r"], errors="coerce").notna()]
summary_rows.append({
    "Approach"     : "6-Model Ensemble (equal weight)",
    "Median_r"     : round(e6d["r"].median(),         3),
    "Median_NSE"   : round(e6d["NSE"].median(),       3),
    "Median_RMSE"  : round(e6d["RMSE"].median(),      3),
    "Median_MAE"   : round(e6d["MAE"].median(),       3),
    "Median_absPBIAS": round(e6d["PBIAS"].abs().median(), 3),
})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_ROOT / "all_approaches_median_summary.csv", index=False)

print("MEDIAN METRIC SUMMARY — ALL THREE APPROACHES")
print("=" * 80)
print(summary_df.to_string(index=False))
print()
print("Interpretation:")
print("  r, NSE         : higher is better (best = 1.0)")
print("  RMSE, MAE      : lower is better  (best = 0.0)")
print("  |PBIAS|        : lower is better  (best = 0.0)")
print()
print("Saved: all_approaches_median_summary.csv")


MEDIAN METRIC SUMMARY — ALL THREE APPROACHES
                         Approach  Median_r  Median_NSE  Median_RMSE  Median_MAE  Median_absPBIAS
                Best Single Model     0.970       0.704        8.506       5.172           21.120
3-Model Ensemble (basin-specific)     0.975       0.565        8.573       5.009           13.328
  6-Model Ensemble (equal weight)     0.966       0.531        8.871       5.193           12.674

Interpretation:
  r, NSE         : higher is better (best = 1.0)
  RMSE, MAE      : lower is better  (best = 0.0)
  |PBIAS|        : lower is better  (best = 0.0)

Saved: all_approaches_median_summary.csv


## 10. Output Summary

In [10]:
print("=" * 72)
print("6-MODEL ENSEMBLE VALIDATION — OUTPUT FILES")
print("=" * 72)
for root, dirs, files in os.walk(OUTPUT_ROOT):
    dirs[:] = sorted(d for d in dirs if not d.startswith("."))
    level   = Path(root).relative_to(OUTPUT_ROOT).parts
    indent  = "  " * len(level)
    print(f"{indent}📁 {Path(root).name}/")
    sub = "  " * (len(level) + 1)
    for f in sorted(files):
        sz = (Path(root) / f).stat().st_size / 1024
        unit = "KB"
        if sz > 1024:
            sz /= 1024; unit = "MB"
        print(f"{sub}📄 {f}  ({sz:.1f} {unit})")

print()
print("=" * 72)
print("ALL VALIDATION NOTEBOOKS COMPLETE")
print("=" * 72)
print()
print("Complete validation pipeline:")
print("  NB03 → Single-model validation     (best model per basin)")
print("  NB07 → 3-model ensemble validation (basin-specific top-3)")
print("  NB10 → 6-model ensemble validation (equal-weight all models)")
print()
print("Final comparison table: all_approaches_median_summary.csv")


6-MODEL ENSEMBLE VALIDATION — OUTPUT FILES
📁 ensemble.6model/
  📄 all_approaches_median_summary.csv  (0.2 KB)
  📄 basin_level_metrics_6ens.csv  (0.9 KB)
  📄 station_level_metrics_6ens.csv  (4.5 KB)
  📄 table_6ens_basin_metrics.csv  (0.7 KB)
  📄 table_6ens_basin_metrics.xlsx  (6.0 KB)
  📄 three_approach_comparison_basin.csv  (1.9 KB)

ALL VALIDATION NOTEBOOKS COMPLETE

Complete validation pipeline:
  NB03 → Single-model validation     (best model per basin)
  NB07 → 3-model ensemble validation (basin-specific top-3)
  NB10 → 6-model ensemble validation (equal-weight all models)

Final comparison table: all_approaches_median_summary.csv
